In [1]:
import pandas as pd

# Load all the excel files:

In [33]:
sheet1 = pd.read_excel("GBP_DataSource1.xlsx", header=0, sheet_name="Orders")
sheet2 = pd.read_excel("GBP_DataSource1.xlsx", header=0, sheet_name="People")
sheet3 = pd.read_excel("GBP_DataSource1.xlsx", header=0, sheet_name="Returns")

# Renaming the coloumn names to make it sql safe
# sheet1.columns = (sheet1.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("/","_"))
# sheet2.columns = (sheet2.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("/","_"))
# sheet3.columns = (sheet3.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("/","_"))

# Ali: Converting anything that isn’t a letter, digit, or underscore into an underscore. It’s more robust for varying input.
def clean_columns(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(r"[^a-z0-9_]", "_", regex=True)
    return df

sheet1 = clean_columns(sheet1)
sheet2 = clean_columns(sheet2)
sheet3 = clean_columns(sheet3)

# Dealing with missing values

In [3]:
sheet1

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country_region,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1.0,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2.0,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3.0,CA-2020-138688,2020-06-12,2020-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4.0,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5.0,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10015,6476.0,CA-2021-140872,2021-06-03,2021-06-10,Standard Class,NR-18550,Nick Radford,Consumer,United States,Pembroke Pines,...,33024.0,South,OFF-BI-10002432,Office Supplies,Binders,Wilson Jones Standard D-Ring Binders,4.5540,3,0.70,-3.4914
10016,6477.0,CA-2021-140872,2021-06-03,2021-06-10,Standard Class,NR-18550,Nick Radford,NaN,United States,Pembroke Pines,...,33024.0,South,OFF-AP-10003622,Office Supplies,Appliances,"Bravo II Megaboss 12-Amp Hard Body Upright, Re...",5.2000,2,0.20,0.5850
10017,6478.0,CA-2021-140872,2021-06-03,2021-06-10,Standard Class,NR-18550,Nick Radford,Consumer,United States,Pembroke Pines,...,33024.0,South,TEC-AC-10003832,Technology,Accessories,Logitech P710e Mobile Speakerphone,205.9920,1,0.20,-2.5749
10018,6479.0,CA-2021-140872,2021-06-03,2021-06-10,NaN,NR-18550,Nick Radford,Consumer,United States,Pembroke Pines,...,33024.0,South,OFF-PA-10000809,Office Supplies,Paper,Xerox 206,15.5520,3,0.20,5.4432


In [6]:
sheet1.isnull().sum()

row_id             5
order_id           2
order_date         2
ship_date          1
ship_mode          1
customer_id        0
customer_name      1
segment            2
country_region     2
city               0
state              1
postal_code       11
region             0
product_id         0
category           0
sub_category       0
product_name       0
sales              0
quantity           0
discount           0
profit             0
dtype: int64

In [7]:
#Imputing row_id with consecutive numbers
# for i in range(1, len(sheet1)-1):
#    if pd.isna(sheet1.loc[i, "row_id"]):
#        prev_val = sheet1.loc[i-1, "row_id"]
#        next_val = sheet1.loc[i+1, "row_id"]
#        if pd.notna(prev_val) and pd.notna(next_val):
#            sheet1.loc[i, "row_id"] = prev_val + 1

# Ali: I adjusted the row ID imputation so it no longer relies on both neighbors being present. 
# Now, it increments from the last valid ID, ensuring all missing values form a continuous sequence." 
last_valid_id = None

for i in range(len(sheet1)):
    if pd.isna(sheet1.loc[i, "row_id"]):
        if last_valid_id is not None:
            sheet1.loc[i, "row_id"] = last_valid_id + 1
            last_valid_id += 1
    else:
        last_valid_id = sheet1.loc[i, "row_id"]

In [8]:
#Imputing Country with mode value and changing all values to US 

mode_v = sheet1["country_region"].mode()[0]
mask = sheet1["country_region"].isna() | (sheet1["country_region"] != mode_v)
changed_rows = sheet1.loc[mask, "country_region"].copy()
sheet1.loc[mask, "country_region"] = mode_v

num_changed = mask.sum()
print("Number of rows changed:", num_changed)

report = pd.DataFrame({"index": changed_rows.index,"previous_value": changed_rows.values,"new_value": mode_v})
print(report)


Number of rows changed: 4
   index previous_value      new_value
0   9338            NaN  United States
1   9348            NaN  United States
2  10012         Canada  United States
3  10013         Canada  United States


In [10]:
# ALI Merging the US postal code csv with our order sheet
# postal_reference = pd.read_csv("zipcodes.csv")
# postal_reference = postal_reference.drop(columns=["county", "timezone", "coordinates"])
# postal_reference.insert(0, "postal_id", range(1, len(postal_reference) + 1))
# postal_reference.insert(4, "country", mode_v)
# postal_reference

# reading zip code reference
zipcodes = pd.read_csv("zipcodes.csv")

# Merge on city and state to align postal codes
merged = pd.merge(sheet1, zipcodes[['city', 'state', 'zip']], on=['city', 'state'], how='left')

# Fill missing postal codes from zip reference
merged['postal_code'] = merged['postal_code'].fillna(merged['zip'])

# Drop the zip column if you no longer need it
final_orders = merged.drop(columns=['zip'])

# Now final_orders has postal codes filled where possible

In [11]:
# Ali
final_orders['postal_code'].isnull().sum()

np.int64(0)

In [12]:
sheet2

,regional_manager,region
0,Sadie Pawthorne,West
1,Chuck Magee,East
2,Roxanne Rodriguez,Central
3,Fred Suzuki,South
4,Roxanne Rodriguez,NaN
5,George Smith,North


In [13]:
sheet2.isnull().sum()

regional_manager    0
region              1
dtype: int64

In [14]:
# Here we decided to delete the row with the duplicate manager name

sheet2 = sheet2.dropna()
sheet2.isnull().sum()

regional_manager    0
region              0
dtype: int64

In [15]:
sheet2

,regional_manager,region
0,Sadie Pawthorne,West
1,Chuck Magee,East
2,Roxanne Rodriguez,Central
3,Fred Suzuki,South
5,George Smith,North


In [16]:
sheet3

,returned,order_id
0,Yes,CA-2018-100762
1,Yes,CA-2018-100762
2,Yes,CA-2018-100762
3,Yes,CA-2018-100762
4,Yes,CA-2018-100867
...,...,...
795,Yes,US-2021-147886
796,Yes,US-2021-147998
797,Yes,US-2021-151127
798,Yes,US-2021-155999


In [17]:
sheet3.isnull().sum()

returned    0
order_id    1
dtype: int64

In [9]:
# Here we decided to delete the row with the missing order id, since we wont be able to get the missing order id

sheet3 = sheet3.dropna()
sheet3.isnull().sum()

returned    0
order_id    0
dtype: int64

In [11]:
#dealing with duplicate data

print(sheet3.shape)
print("Number of dulicates:", sheet3.duplicated().sum())
sheet3[sheet3.duplicated(keep=False)]
sheet3 = sheet3.drop_duplicates()           #keeps the first occurance and deletes the rest
print(sheet3.shape)

(297, 2)
Number of dulicates: 0
(297, 2)


In [12]:
sheet3

,returned,order_id
0,Yes,CA-2018-100762
4,Yes,CA-2018-100867
5,Yes,CA-2018-102652
9,Yes,CA-2018-103373
10,Yes,CA-2018-103744
...,...,...
787,Yes,US-2021-136679
789,Yes,US-2021-147886
796,Yes,US-2021-147998
797,Yes,US-2021-151127


In [19]:
# Ali, should we check for duplicates in sheet 1
# if yes, how can we do that?

# Making the enitity dataframes

In [24]:
print(sheet1.columns)
print(sheet2.columns)
print(sheet3.columns)

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country_region', 'city',
       'state', 'postal_code', 'region', 'product_id', 'category',
       'sub_category', 'product_name', 'sales', 'quantity', 'discount',
       'profit'],
      dtype='object')
Index(['regional_manager', 'region'], dtype='object')
Index(['returned', 'order_id'], dtype='object')


In [29]:
# from sheet1:
orders = sheet1[["order_id", "order_date", "quantity"]]
products = sheet1[["product_id", "category", "sub_category", "product_name"]]
customers = sheet1[["customer_id", "customer_name", "segment"]]
addresses = sheet1[["country_region", "city", "state", "postal_code"]]
revenue = sheet1[["sales", "quantity", "discount", "profit"]] 
shipment = sheet1[["ship_date", "ship_mode"]]

# from sheet 2: 
people = sheet2[["regional_manager"]] #add region_id
regions = sheet2[["region"]] #add region_id

# from sheet 3:
returns = sheet3[["returned", "order_id"]]


In [30]:
orders.isnull().sum()

order_id      2
order_date    2
quantity      0
dtype: int64

In [31]:
# Ali
orders[orders.isnull().any(axis=1)]

,order_id,order_date,quantity
8909,CA-2019-111038,NaT,9
8954,NaN,2019-12-17,2
9038,CA-2019-129525,NaT,13
9217,NaN,2020-10-17,1


# Converting the dataframes into a csv file

In [36]:
with pd.ExcelWriter("cleaned GBP_DataSource.xlsx", engine="openpyxl") as writer:
    orders.to_excel(writer, sheet_name="orders", index=False)
    people.to_excel(writer, sheet_name="People", index=False)
    returns.to_excel(writer, sheet_name="Returns", index=False)
    products.to_excel(writer, sheet_name="products", index=False)
    customers.to_excel(writer, sheet_name="customers", index=False)
    addresses.to_excel(writer, sheet_name="addresses", index=False)
    regions.to_excel(writer, sheet_name="regions", index=False)
    revenue.to_excel(writer, sheet_name="revenue", index=False)
    shipment.to_excel(writer, sheet_name="shipment", index=False)
#    postal_reference.to_excel(writer, sheet_name="addresses", index=False)
# zipcodes.csv is already a csv file